# 🏎️ Phase 1: Strategic Framing & Data Architecture
## Project: ApexMotors Revenue Intelligence System (v3.0)

---

### **Project Overview**
This project develops a predictive intelligence framework to optimize the ApexMotors sales funnel. By leveraging behavioral data from digital touchpoints, we aim to transition from a volume-based lead management approach to a **propensity-driven model** that prioritizes high-value conversion opportunities.

### 🎯 1.1 Problem Understanding & Framing
**Business Context:**
ApexMotors currently faces "Lead Fatigue." While digital marketing efforts generate a high volume of inquiries, the sales team lacks a mechanism to distinguish high-intent buyers from casual "window shoppers." This results in inefficient resource allocation and missed revenue opportunities.

**The Data Science Task:**
This project is framed as a **Binary Classification** task. The objective is to predict the probability of a lead resulting in a `Purchase` (1) vs. `No Purchase` (0) based on real-time behavioral and engagement signals.

**Evaluation Strategy:**
1. **Technical Metric ($F_1$-Score):** Chosen because automotive sales data typically exhibits class imbalance. The $F_1$-Score ensures a balance between **Precision** and **Recall**:
$$F_1 = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$$
2. **Business KPI (Sales Efficiency):** Success is measured by the conversion uplift for prioritized segments and a reduction in manual lead sorting time for the sales department.

In [3]:
import os
import pandas as pd
import numpy as np
from pathlib import Path

# Initialize Root-Relative Portability
ROOT_DIR = Path.cwd()
DATA_DIR = ROOT_DIR / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
MODELS_DIR = ROOT_DIR / "models"
SRC_DIR = ROOT_DIR / "src"

# Create standard directory structure for professional organization (Step 7)
for folder in [RAW_DATA_DIR, PROCESSED_DATA_DIR, MODELS_DIR, SRC_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print(f"✅ Environment initialized at: {ROOT_DIR}")
print(f"📂 Please ensure raw data is placed in: {RAW_DATA_DIR}")

✅ Environment initialized at: /content
📂 Please ensure raw data is placed in: /content/data/raw


### 📖 1.2 Data Dictionary
| Feature | Type | Units/Range | Business Significance |
| :--- | :--- | :--- | :--- |
| **Lead_ID** | Object | Unique ID | Primary key for lead tracking (to be excluded from modeling). |
| **App_Engagement_Mins** | Numerical | 0 - 180 Mins | Quantitative measure of brand immersion. |
| **Web_Configurator_Status**| Binary | 0 or 1 | Indicates active intent to customize a vehicle. |
| **Test_Drive_Completed** | Binary | 0 or 1 | The highest-intent physical behavioral signal. |
| **Last_Contact_Days** | Numerical | 0 - 365 Days | Measures lead "recency" and potential interest decay. |
| **Lead_Source** | Categorical | 5 Channels | Used for evaluating marketing channel ROI. |
| **Purchase** | Binary (Target) | 0 or 1 | The success event (1 = Purchased, 0 = No Purchase). |

### 📥 1.3 Data Acquisition & Instructions
To ensure the reproducibility of this notebook, place the source dataset in the designated raw data directory.

**Instructions:**
1. Locate the file `apex_leads_v3.csv`.
2. Upload or move this file into the following path relative to the notebook:  
   `data/raw/apex_leads_v3.csv`

In [5]:
# Load the dataset from the raw directory
DATA_FILE = RAW_DATA_DIR / "apex_leads_v3.csv"

try:
    df = pd.read_csv(DATA_FILE)
    print("📋 DATA HEALTH AUDIT")
    print("-" * 30)
    print(df.info())

    print("\n⚠️ MISSING VALUE SUMMARY")
    print("-" * 30)
    print(df.isnull().sum())

    print("\n📊 CLASS DISTRIBUTION (TARGET)")
    print("-" * 30)
    # Using 'Purchase' to match the actual column name in the dataset
    print(df['Purchase'].value_counts(normalize=True))

except FileNotFoundError:
    print(f"❌ Error: '{DATA_FILE.name}' not found.")
    print(f"👉 Expected Path: {DATA_FILE}")

📋 DATA HEALTH AUDIT
------------------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Lead_ID                  1200 non-null   object 
 1   App_Engagement_Mins      1140 non-null   float64
 2   Web_Configurator_Status  1200 non-null   int64  
 3   Test_Drive_Completed     1200 non-null   int64  
 4   Last_Contact_Days        1200 non-null   int64  
 5   Lead_Source              1200 non-null   object 
 6   Purchase                 1200 non-null   int64  
dtypes: float64(1), int64(4), object(2)
memory usage: 65.8+ KB
None

⚠️ MISSING VALUE SUMMARY
------------------------------
Lead_ID                     0
App_Engagement_Mins        60
Web_Configurator_Status     0
Test_Drive_Completed        0
Last_Contact_Days           0
Lead_Source                 0
Purchase                    0
dtype: int64

📊 CLASS 

### 🔍 1.4 Initial Audit Observations & Strategic Implications

Analysis of the preliminary health audit identifies three critical technical requirements for the upcoming machine learning lifecycle:

1.  **Missing Value Management (Data Integrity)**:
    * The `App_Engagement_Mins` feature contains **60 null entries** (approximately 5% of the total dataset).
    * **Action**: These will be addressed in the **Step 3: Preprocessing** phase using a median imputation strategy to ensure model robustness without introducing significant variance bias.

2.  **Target Class Imbalance (Evaluation Logic)**:
    * The target variable `Purchase` exhibits a distinct imbalance, with successful conversions representing only **16.92%** of the total records.
    * **Action**: This quantitative evidence confirms that standard "Accuracy" would be a misleading metric. The project will prioritize the **$F_1$-Score** and explore synthetic balancing techniques (e.g., SMOTE) in Phase 4 to ensure the model accurately identifies high-intent leads.

3.  **Feature Encoding Requirements**:
    * Most behavioral predictors are correctly typed as numerical or binary; however, `Lead_Source` is identified as an 'Object' (Categorical).
    * **Action**: Professional encoding (e.g., One-Hot Encoding) will be applied in the feature engineering phase to transform these categories into a format digestible by Gradient Boosting algorithms.